# Stage 05a: Initial Model Training

**Purpose:** Quick validation with default hyperparameters

**Inputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- config_generated/04c_monotonicity_constraints.yaml

**Outputs:**
- models/05a_model_initial.json
- results/05a_predictions_train.parquet
- results/05a_predictions_test.parquet
- results/05a_metrics.yaml

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05a: INITIAL MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()
from model_utils import inverse_transform, predict_with_transform, save_predictions
from data_preparation import prepare_model_data

In [ ]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")

In [ ]:
# Get machine config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]
join_key = cfg["data"]["join_key"]
target_method = cfg["model"].get("target_method", "direct")
print(f"Target method: {target_method}")

In [ ]:
# Prepare data with loss cap filtering
loss_cap = cfg['experiment'].get('loss_cap', None)
loss_type = cfg['experiment'].get('loss_type', 'bi')
exposure_floor = cfg['experiment'].get('exposure_floor', None)

data = prepare_model_data(
    output_base=output_base,
    target_col=target,
    exposure_col=exposure,
    target_method=target_method,
    loss_cap=loss_cap,
    loss_type=loss_type,
    exposure_floor=exposure_floor,
    verbose=True
)

# Extract prepared data
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
w_train = data['w_train']
w_test = data['w_test']
pp_train_actual = data['pp_train']
pp_test_actual = data['pp_test']
glm_train = data['glm_train']
glm_test = data['glm_test']
train_orig = data['train_orig']
test_orig = data['test_orig']
top_100_report = data['top_100_report']

# Save top 100 report if generated
if top_100_report is not None:
    report_path = f"{output_base}/reports"
    import os
    os.makedirs(report_path, exist_ok=True)
    top_100_report.to_csv(f"{report_path}/top_100_targets_before_cap.csv", index=True)
    print(f"  Saved top 100 report to: {report_path}/top_100_targets_before_cap.csv")

print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")

In [ ]:
# Load monotonicity constraints
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
if os.path.exists(mono_file):
    print(f"\n* Loading monotonicity constraints...")
    with open(mono_file, "r") as f:
        mono_dict = yaml.safe_load(f)
    # Build constraints tuple in feature order
    feature_names = X_train.columns.tolist()
    monotone_constraints = tuple(mono_dict.get(f, 0) for f in feature_names)
    n_constrained = sum(1 for c in monotone_constraints if c != 0)
    print(f"  Loaded {n_constrained} constraints")
else:
    monotone_constraints = None
    print(f"\n* No monotonicity constraints found")

In [ ]:
# Build XGBoost parameters
print(f"\n* Building XGBoost parameters...")
xgb_params = cfg["xgboost"].copy()
n_estimators = xgb_params.pop("n_estimators")

# Add monotonicity constraints if available
if monotone_constraints:
    xgb_params["monotone_constraints"] = monotone_constraints

# Fix eval_metric for tweedie
if "eval_metric" in xgb_params and "tweedie" in xgb_params["eval_metric"]:
    if "@" not in xgb_params["eval_metric"]:
        variance_power = xgb_params["tweedie_variance_power"]
        xgb_params["eval_metric"] = f"{xgb_params['eval_metric']}@{variance_power}"

print(f"  n_estimators: {n_estimators}")
print(f"  monotone_constraints: {monotone_constraints is not None}")

In [ ]:
# Train model
print(f"\n* Training XGBoost model...")

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)

model.fit(
    X_train, y_train,
    sample_weight=w_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    sample_weight_eval_set=[w_train, w_test],
    verbose=100
)

print(f"\n* Training complete")
print(f"  Best iteration: {model.best_iteration}")

In [ ]:
# Generate predictions
print(f"\n* Generating predictions...")
# Generate predictions with inverse transform
gbm_train_raw, pred_train = predict_with_transform(model, X_train, glm_train, target_method)
gbm_test_raw, pred_test = predict_with_transform(model, X_test, glm_test, target_method)

print(f"  Predictions: train mean={pred_train.mean():.2f}, test mean={pred_test.mean():.2f}")

# Calculate metrics
mae_train = mean_absolute_error(pp_train_actual, pred_train, sample_weight=w_train)
mae_test = mean_absolute_error(pp_test_actual, pred_test, sample_weight=w_test)
rmse_train = np.sqrt(mean_squared_error(pp_train_actual, pred_train, sample_weight=w_train))
rmse_test = np.sqrt(mean_squared_error(pp_test_actual, pred_test, sample_weight=w_test))

print(f"\n* Metrics:")
print(f"  Train MAE: {mae_train:.4f}, RMSE: {rmse_train:.4f}")
print(f"  Test MAE: {mae_test:.4f}, RMSE: {rmse_test:.4f}")

In [ ]:
# Prepare data for lift charts
import matplotlib.pyplot as plt
from lift_chart_fast import create_lift_chart
from hpo_metrics import compute_fit_quality, compute_model_power
from IPython.display import Image, display

print(f'\n* Preparing lift chart data...')

# Add predictions
train_orig['pred'] = pred_train.values if isinstance(pred_train, pd.Series) else pred_train
test_orig['pred'] = pred_test.values if isinstance(pred_test, pd.Series) else pred_test

# Create incurred columns (pp * ee = capped loss)
train_orig['incurred_act'] = (pp_train_actual if isinstance(pp_train_actual, np.ndarray) else pp_train_actual.values) * w_train
train_orig['incurred_pred'] = (pred_train.values if isinstance(pred_train, pd.Series) else pred_train) * w_train
train_orig['denom'] = 1

test_orig['incurred_act'] = (pp_test_actual if isinstance(pp_test_actual, np.ndarray) else pp_test_actual.values) * w_test
test_orig['incurred_pred'] = (pred_test.values if isinstance(pred_test, pd.Series) else pred_test) * w_test
test_orig['denom'] = 1

# GBM-only columns (to see pure GBM effect - ratios)
train_orig['incurred_act_gbm'] = (y_train if isinstance(y_train, np.ndarray) else y_train.values) * w_train
train_orig['incurred_pred_gbm'] = (gbm_train_raw.values if isinstance(gbm_train_raw, pd.Series) else gbm_train_raw) * w_train

test_orig['incurred_act_gbm'] = (y_test if isinstance(y_test, np.ndarray) else y_test.values) * w_test
test_orig['incurred_pred_gbm'] = (gbm_test_raw.values if isinstance(gbm_test_raw, pd.Series) else gbm_test_raw) * w_test

print(f'  Data prepared (incurred = pp * ee = capped loss)')


In [ ]:
# Train lift chart
print(f'\n* Creating train lift chart...')
fig_train, table_train = create_lift_chart(
    train_orig, 
    exposure,  # weight column for decile binning
    bins=10, 
    title='05a Train Lift Chart'
)

chart_file = f'{output_base}/results/05a_lift_train.png'
fig_train.savefig(chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_train)
print(f'  Saved: {chart_file}')

display(Image(chart_file))
print('\nTrain decile summary:')
print(table_train[['decile', 'n_records', 'ee_bi_imps', 'incurred_act', 'incurred_pred', 'act_ee_weigh_avg', 'act_simple_avg', 'pred_ee_weigh_avg', 'pred_simple_avg']].to_string(index=False))
table_train.to_csv(f"{output_base}/results/05a_lift_train_table.csv", index=False)


In [ ]:
# Test lift chart
print(f'\n* Creating test lift chart...')
fig_test, table_test = create_lift_chart(
    test_orig, 
    exposure,  # weight column for decile binning
    bins=10, 
    title='05a Test Lift Chart'
)

chart_file = f'{output_base}/results/05a_lift_test.png'
fig_test.savefig(chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_test)
print(f'  Saved: {chart_file}')

display(Image(chart_file))
print('\nTest decile summary:')
print(table_test[['decile', 'n_records', 'ee_bi_imps', 'incurred_act', 'incurred_pred', 'act_ee_weigh_avg', 'act_simple_avg', 'pred_ee_weigh_avg', 'pred_simple_avg']].to_string(index=False))
table_test.to_csv(f"{output_base}/results/05a_lift_test_table.csv", index=False)


In [ ]:
# Ratio lift charts (for residual method only)
if target_method == 'residual':
    print(f'\n* Creating ratio lift charts...')
    
    # Train ratio chart
    train_ratio = train_orig.copy()
    train_ratio['incurred_act'] = y_train if isinstance(y_train, np.ndarray) else y_train.values
    train_ratio['incurred_pred'] = gbm_train_raw.values if isinstance(gbm_train_raw, pd.Series) else gbm_train_raw
    train_ratio['denom'] = 1
    
    fig_train_r, table_train_r = create_lift_chart(
        train_ratio, exposure, bins=10,
        title='05a Train Ratio Lift Chart (PP/GLM)'
    )
    chart_file = f'{output_base}/results/05a_lift_train_ratio.png'
    fig_train_r.savefig(chart_file, dpi=150, bbox_inches='tight')
    plt.close(fig_train_r)
    print(f'  Saved: {chart_file}')
    display(Image(chart_file))
    print('\nTrain ratio summary:')
    print(table_train_r[['decile', 'n_records', 'ee_bi_imps', 'incurred_act', 'incurred_pred', 'act_ee_weigh_avg', 'act_simple_avg', 'pred_ee_weigh_avg', 'pred_simple_avg']].to_string(index=False))
    table_train_r.to_csv(f"{output_base}/results/05a_lift_train_ratio_table.csv", index=False)
    
    # Test ratio chart
    test_ratio = test_orig.copy()
    test_ratio['incurred_act'] = y_test if isinstance(y_test, np.ndarray) else y_test.values
    test_ratio['incurred_pred'] = gbm_test_raw.values if isinstance(gbm_test_raw, pd.Series) else gbm_test_raw
    test_ratio['denom'] = 1
    
    fig_test_r, table_test_r = create_lift_chart(
        test_ratio, exposure, bins=10,
        title='05a Test Ratio Lift Chart (PP/GLM)'
    )
    chart_file = f'{output_base}/results/05a_lift_test_ratio.png'
    fig_test_r.savefig(chart_file, dpi=150, bbox_inches='tight')
    plt.close(fig_test_r)
    print(f'  Saved: {chart_file}')
    display(Image(chart_file))
    print('\nTest ratio summary:')
    print(table_test_r[['decile', 'n_records', 'ee_bi_imps', 'incurred_act', 'incurred_pred', 'act_ee_weigh_avg', 'act_simple_avg', 'pred_ee_weigh_avg', 'pred_simple_avg']].to_string(index=False))
    table_test_r.to_csv(f"{output_base}/results/05a_lift_test_ratio_table.csv", index=False)
else:
    print(f'\n* Skipping ratio charts (direct method)')


In [ ]:
# Calculate actuarial metrics
print(f'\n* Calculating actuarial metrics...')

fit_train = compute_fit_quality(pp_train_actual, pred_train, w_train)
fit_test = compute_fit_quality(pp_test_actual, pred_test, w_test)
power_train = compute_model_power(pp_train_actual, pred_train, w_train)
power_test = compute_model_power(pp_test_actual, pred_test, w_test)

print(f'\n* Actuarial Metrics:')
print(f'  Train:')
print(f'    Fit Quality:  {fit_train:.4f}')
print(f'    Model Power:  {power_train:.4f}')
print(f'  Test:')
print(f'    Fit Quality:  {fit_test:.4f}')
print(f'    Model Power:  {power_test:.4f}')

In [ ]:
# Save model
os.makedirs(f"{output_base}/models", exist_ok=True)
model_file = f"{output_base}/models/05a_model_initial.json"
model.get_booster().save_model(model_file)
print(f"\n* Saved model: {model_file}")

In [ ]:
# Save predictions using utility function
print(f"\n* Saving predictions...")
save_predictions(output_base, "05a", "train", pp_train_actual, pred_train, w_train)
save_predictions(output_base, "05a", "test", pp_test_actual, pred_test, w_test)
print("  ✓ Saved")

In [ ]:
# Save metrics
metrics = {
    "stage": "05a_initial",
    "train": {"mae": float(mae_train), "rmse": float(rmse_train)},
    "test": {"mae": float(mae_test), "rmse": float(rmse_test)},
    "best_iteration": int(model.best_iteration),
    "n_features": int(X_train.shape[1]),
    "monotone_constraints_applied": monotone_constraints is not None,
}

with open(f"{output_base}/results/05a_metrics.yaml", "w") as f:
    yaml.dump(metrics, f)

print(f"* Saved metrics")

In [ ]:
# Create analysis table
print(f'\n* Creating analysis table...')

from model_utils import create_model_analysis_table, create_decile_analysis_table

analysis_test = create_model_analysis_table(
    df=test_orig,
    target=y_test,
    pred_gbm=gbm_test_raw,
    glm_pred=glm_test,
    exposure=w_test,
    pp_actual_from_capped=pp_test_actual,
    pred_pp=pred_test
)

decile_analysis_test = create_decile_analysis_table(analysis_test)

report_path = f'{output_base}/reports'
import os
os.makedirs(report_path, exist_ok=True)
decile_analysis_test.to_csv(f'{report_path}/05a_decile_analysis_test.csv', index=False)
print(f'  Saved: {report_path}/05a_decile_analysis_test.csv')

print(f'\nTest Decile Analysis:')
print(decile_analysis_test.to_string(index=False))


In [ ]:
print("\n########################################")
print("# STAGE 05a: COMPLETE")
print("########################################")